# TUNGAR-Guard — Veri Seti + YOLOv8 Eğitimi

4 kaynağı otomatik indirip birleştiriyor, key/login gerekmiyor:

1. `Chapian/PPE_detection` → person, hardhat, no_hardhat, vest, no_vest, gloves, no_gloves, boots, no_boots
2. `keremberke/forklift-object-detection` → forklift
3. `keremberke/construction-safety-object-detection` → excavators, dump truck, mini-van, truck, wheel loader, barricade, dumpster, safety net, safety shoes
4. `keremberke/smoke-object-detection` → smoke (21k+ görsel var, dengeyi bozmasın diye örneklem sınırlı alınıyor)

Kutu/koli ve üretim hattı makineleri için temiz bir kaynak bulamadım, o yüzden yok.

Kapsam dışı: yaralanma/kan görüntüsü ve hırsızlık — ikisi de bu notebook'un işi değil, VLM katmanının işi.

Son adımda dataset zip'i ve eğitilmiş model Drive'a kaydediliyor.

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q ultralytics huggingface_hub pyyaml

## 1) Yardımcı fonksiyonlar

In [ ]:
import os, glob, shutil, zipfile, yaml, json, random
from huggingface_hub import snapshot_download

MERGED_ROOT = "/content/merged_yolo"
for split in ["train", "val"]:
    os.makedirs(f"{MERGED_ROOT}/{split}/images", exist_ok=True)
    os.makedirs(f"{MERGED_ROOT}/{split}/labels", exist_ok=True)

unified_names = []
name_to_idx = {}

# tire/alt çizgi gibi yazım farkı yüzünden ayrı sayılan ama aynı anlama gelen sınıflar
CLASS_ALIASES = {
    "no-hardhat": "no_hardhat",
}

DROP_CLASSES = {"gloves", "no-mask", "safety shoes", "no-safety vest", "safety vest"}

def normalize_class_name(name):
    key = name.strip().lower()
    return CLASS_ALIASES.get(key, key)

def get_or_add(name):
    key = normalize_class_name(name)
    if key not in name_to_idx:
        name_to_idx[key] = len(unified_names)
        unified_names.append(key)
    return name_to_idx[key]

def find_yolo_export(root_dir):
    yaml_candidates = glob.glob(f"{root_dir}/**/data.yaml", recursive=True) + glob.glob(f"{root_dir}/**/*.yaml", recursive=True)
    if not yaml_candidates:
        return None
    yp = yaml_candidates[0]
    with open(yp) as f:
        cfg = yaml.safe_load(f)
    base = os.path.dirname(yp)
    return cfg, base

def find_class_names(split_dir, fallback=["object"]):
    yaml_candidates = glob.glob(f"{split_dir}/**/*.yaml", recursive=True)
    if yaml_candidates:
        with open(yaml_candidates[0]) as f:
            cfg = yaml.safe_load(f)
        if cfg and "names" in cfg:
            return cfg["names"]
    txt_candidates = (glob.glob(f"{split_dir}/**/_classes.txt", recursive=True)
                       + glob.glob(f"{split_dir}/**/classes.txt", recursive=True))
    if txt_candidates:
        with open(txt_candidates[0]) as f:
            names = [l.strip() for l in f if l.strip()]
        if names:
            return names
    return fallback

def download_zip_dataset(repo_id, extract_root):
    # roboflow2huggingface zip export'ları (data/*.zip); etiketler txt ya da coco json olabilir
    repo_dir = snapshot_download(repo_id=repo_id, repo_type="dataset",
                                  allow_patterns=["data/*.zip", "*.yaml", "*.txt"])
    os.makedirs(extract_root, exist_ok=True)
    zip_files = glob.glob(f"{repo_dir}/**/*.zip", recursive=True)
    splits = {}
    for zp in zip_files:
        split_name = os.path.splitext(os.path.basename(zp))[0]
        out_dir = f"{extract_root}/{split_name}"
        os.makedirs(out_dir, exist_ok=True)
        with zipfile.ZipFile(zp) as z:
            z.extractall(out_dir)
        img_dirs = glob.glob(f"{out_dir}/**/images", recursive=True)
        img_dir = img_dirs[0] if img_dirs else out_dir
        names = find_class_names(out_dir)
        n_imgs = len(glob.glob(f"{img_dir}/*.jpg") + glob.glob(f"{img_dir}/*.png") + glob.glob(f"{img_dir}/*.jpeg"))
        splits[split_name] = {"images": img_dir, "label_root": out_dir, "names": names, "count": n_imgs}
        print(f"  [{repo_id} / {split_name}] {n_imgs} görsel, sınıflar={names}")
    return splits

def build_label_index(label_root, src_names):
    index = {}
    txt_files = [t for t in glob.glob(f"{label_root}/**/*.txt", recursive=True)
                 if os.path.basename(t).lower() not in ("classes.txt", "_classes.txt")]

    if txt_files:
        for lp in txt_files:
            stem = os.path.splitext(os.path.basename(lp))[0]
            boxes = []
            with open(lp) as f:
                for line in f:
                    parts = line.split()
                    if len(parts) < 5:
                        continue
                    try:
                        cls_idx = int(float(parts[0]))
                        xc, yc, bw, bh = map(float, parts[1:5])
                    except ValueError:
                        continue
                    if cls_idx < 0 or cls_idx >= len(src_names):
                        continue
                    boxes.append((src_names[cls_idx], xc, yc, bw, bh))
            index[stem] = boxes
        return index

    # txt yoksa coco json dene
    coco_files = glob.glob(f"{label_root}/**/*.json", recursive=True)
    for cf in coco_files:
        try:
            with open(cf) as f:
                coco = json.load(f)
            if not all(k in coco for k in ("images", "annotations", "categories")):
                continue
        except Exception:
            continue
        images_by_id = {img["id"]: img for img in coco["images"]}
        cats_by_id = {c["id"]: c["name"] for c in coco["categories"]}
        for ann in coco["annotations"]:
            img = images_by_id.get(ann["image_id"])
            if not img:
                continue
            stem = os.path.splitext(os.path.basename(img["file_name"]))[0]
            w_img, h_img = img.get("width"), img.get("height")
            if not w_img or not h_img:
                continue
            x, y, bw, bh = ann["bbox"]
            xc = (x + bw / 2) / w_img
            yc = (y + bh / 2) / h_img
            nw = bw / w_img
            nh = bh / h_img
            name = cats_by_id.get(ann["category_id"], "object")
            index.setdefault(stem, []).append((name, xc, yc, nw, nh))
        if index:
            print(f"  (COCO json kullanıldı: {cf})")
            return index
    return index

def copy_split(src_images_dir, label_root, src_names, dst_split, prefix, max_images=None):
    label_index = build_label_index(label_root, src_names)
    img_files = (glob.glob(f"{src_images_dir}/*.jpg") + glob.glob(f"{src_images_dir}/*.jpeg")
                 + glob.glob(f"{src_images_dir}/*.png"))
    if max_images is not None and len(img_files) > max_images:
        random.seed(42)
        img_files = random.sample(img_files, max_images)
    copied, with_labels = 0, 0
    for img_path in img_files:
        base = os.path.basename(img_path)
        stem = os.path.splitext(base)[0]
        boxes = label_index.get(stem, [])
        new_base = f"{prefix}_{base}"
        shutil.copy(img_path, f"{MERGED_ROOT}/{dst_split}/images/{new_base}")
        dst_lbl = f"{MERGED_ROOT}/{dst_split}/labels/{prefix}_{stem}.txt"
        lines_out = []
        for name, xc, yc, bw, bh in boxes:
            if normalize_class_name(name) in DROP_CLASSES:
                continue
            new_cls = get_or_add(name)
            lines_out.append(f"{new_cls} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}")
        with open(dst_lbl, "w") as f:
            f.write("\n".join(lines_out))
        if lines_out:
            with_labels += 1
        copied += 1
    print(f"  [{prefix} -> {dst_split}] {copied} görsel kopyalandı, {with_labels} tanesinde etiket bulundu"
          + ("  ⚠️ HİÇ ETİKET BULUNAMADI!" if copied > 0 and with_labels == 0 else ""))
    return copied

def resolve(base, cfg, key, fallback_dirname):
    val = cfg.get(key)
    if val:
        cand = os.path.normpath(os.path.join(base, val))
        if os.path.isdir(cand):
            return cand
    cand = os.path.join(base, fallback_dirname, "images")
    return cand if os.path.isdir(cand) else None

print("Hazır.")

## 2) Kaynak 1 — PPE dataset'i

In [ ]:
ppe_dir = snapshot_download(repo_id="Chapian/PPE_detection", repo_type="dataset",
                             ignore_patterns=["*.pt", "*.cache"])
print("PPE indirildi:", ppe_dir)

ppe_export = find_yolo_export(ppe_dir)
assert ppe_export, "PPE data.yaml bulunamadı."
ppe_cfg, ppe_base = ppe_export
ppe_names = ppe_cfg["names"]

ppe_train_img = resolve(ppe_base, ppe_cfg, "train", "train")
ppe_val_img = resolve(ppe_base, ppe_cfg, "val", "valid") or resolve(ppe_base, ppe_cfg, "val", "val")
assert ppe_train_img, "PPE train klasörü bulunamadı"

ppe_train_label_root = os.path.dirname(ppe_train_img)
ppe_val_label_root = os.path.dirname(ppe_val_img) if ppe_val_img else None

print("PPE sınıfları:", ppe_names)
print("PPE train:", ppe_train_img)
print("PPE val:  ", ppe_val_img)

## 3) Kaynak 2 — Forklift dataset'i

In [ ]:
forklift_splits = download_zip_dataset("keremberke/forklift-object-detection", "/content/forklift_raw")

## 4) Kaynak 3 — İnşaat/depo araçları

In [ ]:
construction_splits = download_zip_dataset("keremberke/construction-safety-object-detection", "/content/construction_raw")

## 4b) Kaynak 4 — Yangın/duman dataset'i

21k+ görsel var, örneklem sınırlı alınacak (~300 train / 60 val).

In [ ]:
smoke_splits = download_zip_dataset("keremberke/smoke-object-detection", "/content/smoke_raw")

## 5) Kaynakları birleştir

In [ ]:
print("PPE:")
copy_split(ppe_train_img, ppe_train_label_root, ppe_names, "train", "ppe")
if ppe_val_img:
    copy_split(ppe_val_img, ppe_val_label_root, ppe_names, "val", "ppe")

print("\nForklift:")
for split_key, info in forklift_splits.items():
    dst = "train" if split_key == "train" else "val"
    copy_split(info["images"], info["label_root"], info["names"], dst, f"fork_{split_key}")

print("\nİnşaat/depo araçları:")
for split_key, info in construction_splits.items():
    dst = "train" if split_key == "train" else "val"
    copy_split(info["images"], info["label_root"], info["names"], dst, f"cons_{split_key}")

print("\nYangın/duman (örneklem sınırlı):")
SMOKE_MAX = {"train": 300, "valid": 60, "test": 60}
for split_key, info in smoke_splits.items():
    dst = "train" if split_key == "train" else "val"
    copy_split(info["images"], info["label_root"], info["names"], dst, f"smoke_{split_key}",
               max_images=SMOKE_MAX.get(split_key, 60))

print("\nBirleşik sınıf listesi:", unified_names)
print("Toplam sınıf sayısı:", len(unified_names))

In [ ]:
class_names = unified_names

data_cfg = {
    "train": f"{MERGED_ROOT}/train/images",
    "val": f"{MERGED_ROOT}/val/images",
    "nc": len(class_names),
    "names": class_names,
}
data_yaml_path = "/content/data_merged.yaml"
with open(data_yaml_path, "w") as f:
    yaml.safe_dump(data_cfg, f)

print(open(data_yaml_path).read())
print(f"train: {len(glob.glob(f'{MERGED_ROOT}/train/images/*'))} görsel")
print(f"val:   {len(glob.glob(f'{MERGED_ROOT}/val/images/*'))} görsel")

## 6) Veriyi doğrula — örnek etiketli görseller

In [ ]:
import cv2, random
import matplotlib.pyplot as plt

train_img_dir = f"{MERGED_ROOT}/train/images"
train_lbl_dir = f"{MERGED_ROOT}/train/labels"
all_imgs = glob.glob(f"{train_img_dir}/*")
sample_imgs = random.sample(all_imgs, min(6, len(all_imgs)))

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for ax, img_path in zip(axes.flat, sample_imgs):
    img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    stem = os.path.splitext(os.path.basename(img_path))[0]
    lbl_path = os.path.join(train_lbl_dir, stem + ".txt")
    if os.path.exists(lbl_path):
        with open(lbl_path) as f:
            for line in f:
                parts = line.split()
                if len(parts) < 5:
                    continue
                cls, xc, yc, bw, bh = map(float, parts[:5])
                x1, y1 = int((xc - bw/2) * w), int((yc - bh/2) * h)
                x2, y2 = int((xc + bw/2) * w), int((yc + bh/2) * h)
                cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 0), 2)
                cv2.putText(img, class_names[int(cls)], (x1, max(y1-5, 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)
    ax.imshow(img)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 7) Eğitim

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")  # yolov8n yerine: daha fazla kapasite, person gibi çeşitliliği zor sınıflarda yardımcı oluyor

results = model.train(
    data=data_yaml_path,
    epochs=80,
    imgsz=640,
    batch=16,
    patience=20,
    project="runs_mini_test",
    name="yolov8n_minitest",
    exist_ok=True,
    seed=42,
    verbose=True
)

train_save_dir = str(results.save_dir)
run_dir = train_save_dir
best_weights = f"{run_dir}/weights/best.pt"
print("\nEğitim çıktıları burada:", train_save_dir)

## 8) Değerlendirme — mAP, precision, recall

In [ ]:
print("Kullanılan ağırlık:", best_weights)
best_model = YOLO(best_weights)
metrics = best_model.val(data=data_yaml_path)

print(f"mAP50:    {metrics.box.map50:.3f}")
print(f"mAP50-95: {metrics.box.map:.3f}")
print(f"Precision:{metrics.box.mp:.3f}")
print(f"Recall:   {metrics.box.mr:.3f}")

print("\nSınıf bazlı mAP50:")
# ap50 dizisi class_names ile aynı sırada değil, sadece val'de örneği olan sınıfları içeriyor
if hasattr(metrics.box, "ap_class_index") and len(metrics.box.ap_class_index) > 0:
    for cls_idx, ap in zip(metrics.box.ap_class_index, metrics.box.ap50):
        print(f"  {class_names[int(cls_idx)]}: {ap:.3f}")
    evaluated = {class_names[int(i)] for i in metrics.box.ap_class_index}
    missing = [n for n in class_names if n not in evaluated]
    if missing:
        print(f"\n(val setinde örneği olmadığı için değerlendirilemeyen sınıflar: {missing})")
else:
    print("  (val setinde hiçbir sınıf için örnek bulunamadı)")

In [ ]:
from IPython.display import Image, display
display(Image(filename=f"{run_dir}/results.png"))

## 9) Örnek tahminleri görsel olarak kontrol et

In [ ]:
pred_source = f"{MERGED_ROOT}/val/images"
pred_results = best_model.predict(source=pred_source, conf=0.4, imgsz=640, save=True,
                                   project="runs_mini_test", name="predictions", exist_ok=True)

pred_save_dir = str(pred_results[0].save_dir)
print("Tahminler burada kaydedildi:", pred_save_dir)
pred_imgs = glob.glob(f"{pred_save_dir}/*.jpg")[:6]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for ax, img_path in zip(axes.flat, pred_imgs):
    img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 10) Dataset zip'ini ve eğitilmiş modeli Drive'a kaydet

Colab kapanınca `/content` silinir, bu hücreyi çalıştırmadan çıkmayın.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_FOLDER = "/content/drive/MyDrive/TUNGAR-Guard"
os.makedirs(DRIVE_FOLDER, exist_ok=True)

shutil.copy(data_yaml_path, f"{MERGED_ROOT}/data.yaml")
zip_path = shutil.make_archive('/content/tungar_guard_yolo_dataset', 'zip', MERGED_ROOT)
print("Zip oluşturuldu:", zip_path, f"({round(os.path.getsize(zip_path)/1e6, 1)} MB)")

shutil.copy(zip_path, f"{DRIVE_FOLDER}/tungar_guard_yolo_dataset.zip")
shutil.copy(best_weights, f"{DRIVE_FOLDER}/tungar_guard_yolov8n.pt")

print(f"\n✅ Kaydedildi: {DRIVE_FOLDER}/tungar_guard_yolo_dataset.zip")
print(f"✅ Kaydedildi: {DRIVE_FOLDER}/tungar_guard_yolov8n.pt")